In [ ]:
#LM Studio 세팅

##INFO
#모델: LGAI-EXAONE/EXAONE-3.5-32B-Instruct-GGUF
#파일: EXAONE-3.5-32B-Instruct-Q4_K_M.gguf
#This model's API identifier: exaone-3.5-32b-instruct

##Context
#프리셋: 프레임분석

#Inference
#온도 0.0
#Top-k: 40
#반복 패널티: 1
#Min p 샘플링: 0.05
#Top-p: 0.95
#구조화된 출력(Structed Output-valid Json):
{
  "type": "object",
  "properties": {
    "frame": {
      "type": "string",
      "description": "AI Risk 프레임 이름"
    },
    "reason": {
      "type": "string",
      "description": "분류한 근거와 이유"
    }
  },
  "required": [
    "frame",
    "reason"
  ]
}

##Load
#Context Length: 4096
#GPU Offload: 34/64
#CPU Thread Pool size: 4
#Evaluation Batch size: 512

>>

인과분류 Json Scheme 만들기. 2025.08.13

lable=3 object and timing and
value = 9

Label square

In [ ]:
# 프레임 분석 - EXAONE 32B GGUF + LM Studio 버전
import pandas as pd
import openai
from tqdm import tqdm
import re
import os
import logging
import time
import json

In [ ]:
# 로깅 설정
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler("ai_risk_frame_analysis.log"),
              logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

In [ ]:
#프레임 변인과 조작적 정의
ALLOWED_FRAMES = {
    "차별 및 유해성(Discrimination & toxicity)": {
        "description": "AI가 개인이나 특정 그룹에게 불공정한 응답을 생성하거나, 혐오, 폭력, 불법 행위, 인종차별 등 인간을 유해하거나 부적절한 콘텐츠에 노출시키는 위험을 말한다. 차별 및 유해성을 구성하는 하위 위험 영역은 가. 불공정한 차별, 나. 유해 콘텐츠 노출, 다. 집단 간 불평등한 성능이 포함된다.",
    },
    "프라이버시 및 보안(Privacy & security)": {
        "description": "AI가 개인정보를 무단으로 수집, 유출하거나 추론하여 인간의 사생활을 침해하는 위험 또는 AI의 보안 취약점이 악용되어 데이터 침해나 시스템 조작을 유발하는 위험을 말한다. 가. 민감한 정보를 획득, 유출, 추론하여 개인정보 침해, 나. AI 시스템의 보안 취약점과 이를 공격하는 위험을 포함한다.",
    },
    "오정보(misinformation)": {
        "description": "AI가 사실과 다르거나 오해의 소지가 있는 정보를 의도치 않게 생성하거나 확산시켜 사용자의 잘못된 신념을 형성하게 하는 위험. AI에 의한 '필터버블', 현실 인식의 왜곡, 사회적 결속이 약화되는 위험. 가. 허위정보 또는 오해의 소지가 있는 정보의 생산, 정보 생태계의 오염 및 합의 현실의 상실이 유발될 수 있다.",
    },
    "악의적 행위자의 오용(malicious actors & misuse)": {
        "description": "AI가 악의적인 행위자에 의해 대규모 허위 정보 유포, 감시, 조작, 사이버 공격, 무기 개발/사용 등 해로운 목적으로 사용되어 대중에게 해를 끼치는 위험. 사기, 해킹, 등 개인적 이득을 위한 오용의 위험이다. 악의적인 목적을 가진 인간이 AI를 활용하여 가. 대규모 허위조작정보 및 감시, 나. 사이버 공격, 무기 개발, 또는 사용 그리고 대량 살상을 설계, 다. 사기, 스캠 및 표적 조작하는 위험을 포함한다.",
    },
    "인간-컴퓨터 상호작용(human-computer interaction)": {
        "description": "인간이 AI에 과도하게 의존하거나 AI를 의인화하여 부적절한 관계나 기대를 형성하는 위험, 또는 AI에 중요한 결정을 과도하게 위임하여 인간의 주체성 및 자율성이 저하되는 위험을 다룬다. 가. 과도한 AI 의존 및 안전하지 않은 사용, 나. 인간의 주체성과 자율성 사실의 위험을 포함한다.",
    },
    "사회경제적 및 환경적(socioeconomic & environmental)": {
        "description": "AI로 인해 권력과 자원이 특정 그룹에 집중되어 불평등 및 사회 양극화가 심화되거나, 일자리 자동화로 고용의 질이 저하되고 인간 노동의 경제적, 문화적 가치가 하락하는 위험을 포함한다. 가. 권력 집중 및 불공정한 이익 분배. 나. 불평등 심화 및 고용의 질 저하. 다. 인간 노력의 경제적, 라. 문화적 가치 절하. 경쟁 역학. 마. 거버넌스 실패. 환경적 피해가 포함된다.",
    },
    "AI 시스템 안전, 실패 및 한계(AI system safety, failures, & limitations)": {
        "description": "AI 시스템이 인간의 목표나 가치와 충돌하는 방식으로 행동하거나(오정렬의 문제), 기만, 통제 불가능, 해석 불가능 등 위험한 능력을 갖게 되는 위험을 다룬다. 또한 시스템의 신뢰성 및 견고성 부족으로 인한 오류 및 실패 위험, 의사 결정 과정의 해석 불가능성의 문제를 다룬다. 구체적으로는 가. 인간 목표 또는 가치와 상충하는 자체적 목표를 추구하는 AI, 나. 위험한 기능을 가진 AI, 다. 능력이나 견고성의 부족, 라. 투명성, 해석가능성 부족, 마. AI의 복지 및 권리, 바. 멀티 에이전트 위험을 포함한다.",
    },
    "관련없음": {# 중요: 관련없음 옵션 추가
        "description": "Ai Risk와 직접적인 관련이 없거나, 위 7개 프레임 중 어느 것에도 해당하지 않는 text",
    }
}

def format_allowed_frames():
    """AI Risk에 특화된 프레임 목록을 프롬프트용 문자열로 변환"""
    frame_list = []
    for frame_name, frame_info in ALLOWED_FRAMES.items():
        frame_text = f"""- {frame_name}: {frame_info['description']}"""
        frame_list.append(frame_text)
    return "\n".join(frame_list)

#JSON 형식 프롬프트 템플릿
PROMPT_TEMPLATE = """다음 텍스트를 8개 AI RISK 프레임 중 하나로 분류해주세요.

[AI 관련 텍스트]: {text}

[AI RISK 프레임 목록]:
{allowed_frames}

위 8개 AI RISK 프레임 중 가장 적절한 하나를 선택하고, 그 이유를 설명하세요.

**중요한 분류 기준:**
- AI 관련 텍스트가 AI RISK와 직접적인 관련이 없다면 "관련없음"을 선택하세요
- AI 관련 텍스트가 7개 프레임 중 어느 것에도 명확하게 부합하지 않는다면 "관련없음"을 선택하세요
- 억지로 프레임에 끼워맞추지 마세요

**반드시 다음 JSON 형식으로만 답변하세요:**
{{
  "frame": "정확한 프레임 이름 (위 목록에서 선택)",
  "reason": "해당 프레임으로 분류한 근거와 이유"
}}

답변 주의사항:
- frame과 reason 모두 필수
- 특수문자나 줄바꿈 사용 금지
- JSON 외 다른 내용 금지

JSON 응답:"""

def clean_response_control_chars(response):
    """LLM 응답에서 제어문자 제거 - 강화 버전"""
    if not response:
        return ""

    # 문자열로 변환
    response = str(response)

    # 제어문자 제거 (JSON에 문제가 되는 것들)
    response = response.replace('\n', ' ')       # 줄바꿈 → 공백
    response = response.replace('\r', ' ')       # 캐리지 리턴 → 공백
    response = response.replace('\t', ' ')       # 탭 → 공백
    response = response.replace('\b', ' ')       # 백스페이스 → 공백
    response = response.replace('\f', ' ')       # 폼피드 → 공백
    response = response.replace('\v', ' ')       # 수직 탭 → 공백
    response = response.replace('\0', '')        # NULL 문자 제거

    # 기타 문제가 되는 제어문자들 제거 (ASCII 0-31, 127-159)
    response = re.sub(r'[\x00-\x1f\x7f-\x9f]', ' ', response)

    # 연속된 공백 정리
    response = re.sub(r'\s+', ' ', response).strip()

    return response

def clean_json_content(json_text):
    """JSON 내용 안의 문자열 값들도 정리"""
    try:
        # JSON 파싱 전에 내용 정리
        json_text = clean_response_control_chars(json_text)

        # JSON 파싱
        result = json.loads(json_text)

        # frame과 reason 값도 정리
        if 'frame' in result:
            result['frame'] = clean_response_control_chars(result['frame'])
        if 'reason' in result:
            result['reason'] = clean_response_control_chars(result['reason'])

        return result
    except:
        return None

def extract_frame_from_text(response):
    """JSON 파싱 실패 시 텍스트에서 직접 프레임 추출"""
    try:
        cleaned_response = clean_response_control_chars(response)

        # 텍스트에서 프레임 찾기
        for frame_name in ALLOWED_FRAMES.keys():
            # 전체 프레임명 검색
            if frame_name in cleaned_response:
                return frame_name, "텍스트 파싱으로 추출"

            # 프레임명의 한국어 부분만 검색 (관련없음 제외)
            if frame_name != "관련없음":
                korean_part = frame_name.split('(')[0].strip()
                if korean_part in cleaned_response:
                    return frame_name, "텍스트 파싱으로 추출"

        return "관련없음", "프레임 추출 실패"

    except Exception as e:
        logger.error(f"텍스트 파싱 중 오류: {str(e)}")
        return "관련없음", f"텍스트 파싱 오류: {str(e)}"

def extract_frame_from_json(response):
    """JSON에서 프레임과 이유 추출 - 강화된 제어문자 처리"""
    try:
        # 1단계: 기본 응답 정리
        cleaned_response = clean_response_control_chars(response)

        # 2단계: JSON 부분만 추출 (앞뒤 텍스트 제거)
        json_match = re.search(r'\{.*?\}', cleaned_response, re.DOTALL)
        if json_match:
            json_text = json_match.group(0)

            # 3단계: 강화된 JSON 파싱
            result = clean_json_content(json_text)

            if result is None:
                logger.warning("JSON 파싱 실패, 텍스트 파싱으로 전환")
                return extract_frame_from_text(cleaned_response)

            # 4단계: 결과 추출 및 검증
            frame = result.get("frame", "").strip()
            reason = result.get("reason", "").strip()

            # 5단계: 프레임 이름 검증 및 정규화
            if frame not in ALLOWED_FRAMES:
                # 부분 매칭으로 프레임 찾기
                for allowed_frame in ALLOWED_FRAMES:
                    if allowed_frame == "관련없음":
                        continue  # 관련없음은 정확히 매칭되어야 함

                    if (allowed_frame.lower() in frame.lower() or
                        frame.lower() in allowed_frame.lower() or
                        allowed_frame.split('(')[0].strip().lower() in frame.lower()):
                        frame = allowed_frame
                        break
                else:
                    # 아무것도 매치되지 않으면 관련없음으로 분류
                    frame = "관련없음"
                    reason = f"허용되지 않은 프레임이므로 관련없음으로 분류: {frame}"

            return frame, reason

        else:
            # JSON 형식을 찾지 못한 경우 텍스트에서 직접 추출 시도
            return extract_frame_from_text(cleaned_response)

    except json.JSONDecodeError as e:
        logger.warning(f"JSON 파싱 오류: {str(e)}")
        logger.warning(f"문제가 된 응답: {response[:200]}...")

        # JSON 파싱 실패 시 텍스트 파싱으로 폴백
        return extract_frame_from_text(response)

    except Exception as e:
        logger.error(f"JSON 추출 중 오류: {str(e)}")
        return "관련없음", f"추출 오류: {str(e)}"

In [ ]:
class NewsFrameAnalyzer:
    def __init__(self, client, output_dir='./'):
        """
        EXAONE 32B GGUF 버전 초기화
        client: OpenAI client (LM Studio 연결)
        """
        self.client = client
        self.output_dir = output_dir

        # 출력 디렉토리 생성
        os.makedirs(self.output_dir, exist_ok=True)

        logger.info("🚀 EXAONE 32B GGUF 프레임 분석기 초기화 완료!")

    def analyze_news(self, text):
        """뉴스 텍스트를 JSON 형식으로 분석 - GGUF 버전"""
        # JSON 프롬프트 구성
        allowed_frames_text = format_allowed_frames()
        prompt = PROMPT_TEMPLATE.format(text=text, allowed_frames=allowed_frames_text)

        try:
            # EXAONE 32B로 추론 수행
            response = self.client.chat.completions.create(
                model="exaone-3.5-32b-instruct",
                messages=[
                    {"role": "system", "content": "당신은 AI RISK를 정확히 분석하는 컴퓨터공학자입니다. 주어진 8개 프레임 중 텍스트에 해당하는 하나를 선택하여 JSON 형식으로 정확히 응답하세요."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.0,        # 일관된 분석을 위해 낮게
                max_tokens=512,         # JSON 응답에 충분한 길이
                #top_p=0.9,             # 다양성 조절
                frequency_penalty=0.0,
                presence_penalty=0.0
            )

            # 응답 추출
            model_response = response.choices[0].message.content.strip()

            # 디버깅용 로그
            logger.debug(f"EXAONE 32B 응답: {model_response}")

            # JSON에서 프레임 및 이유 추출
            frame, reason = extract_frame_from_json(model_response)

            # 결과 로그
            logger.info(f"추출된 프레임: {frame}, 이유: {reason}")

            return frame, reason

        except Exception as e:
            logger.error(f"EXAONE 32B 분석 중 오류: {str(e)}")
            return "관련없음", f"분석 오류: {str(e)}"

    def process_dataframe(self, df, output_file='AiRisk_frame_analysis_EXAONE32B_results_test3.csv'):#checkpoint와 result를 내보내기할 폴더경로.
        """데이터프레임 처리 - GGUF 32B 버전"""
        # 파일 경로 설정
        output_path = os.path.join(self.output_dir, output_file)
        checkpoint_path = os.path.join(self.output_dir, f"checkpoint_{output_file}")

        # 결과 저장용 리스트
        frame_names = []
        frame_reasons = []

        # 체크포인트 복구
        start_idx = 0
        if os.path.exists(checkpoint_path):
            try:
                checkpoint_df = pd.read_csv(checkpoint_path, encoding='utf-8-sig')
                if '프레임 이름' in checkpoint_df.columns and '이유' in checkpoint_df.columns:
                    completed_rows = sum(~checkpoint_df['프레임 이름'].isna())
                    if completed_rows > 0:
                        start_idx = completed_rows
                        logger.info(f"체크포인트에서 복구: {start_idx}행부터 계속")
                        frame_names = checkpoint_df['프레임 이름'].tolist()[:start_idx]
                        frame_reasons = checkpoint_df['이유'].tolist()[:start_idx]
            except Exception as e:
                logger.warning(f"체크포인트 복구 실패: {str(e)}")

        # 진행 상황 표시
        pbar = tqdm(total=len(df) - start_idx, initial=0, desc="🚀 EXAONE 32B GGUF 프레임 분석")

        # 데이터 순차 처리
        for i in range(start_idx, len(df)):
            # 텍스트 가져오기
            text = df.iloc[i]['text']

            # 누락된 데이터 처리
            if pd.isna(text) or text == "":
                text = "본문 정보 없음"

            try:
                # 뉴스 분석 (최대 3번 재시도)
                for attempt in range(3):
                    try:
                        frame, reason = self.analyze_news(text)
                        break
                    except Exception as e:
                        if attempt < 2:  # 3번 중 2번까지는 재시도
                            logger.warning(f"행 {i} 처리 실패 (시도 {attempt+1}/3): {str(e)}")
                            time.sleep(1)  # LM Studio는 GPU 메모리 정리 불필요
                        else:
                            logger.error(f"행 {i} 최대 재시도 초과: {str(e)}")
                            frame, reason = "관련없음", f"처리 실패: {str(e)}"
            except Exception as e:
                logger.error(f"행 {i} 처리 중 예외 발생: {str(e)}")
                frame, reason = "관련없음", f"처리 오류: {str(e)}"

            # 결과 저장
            frame_names.append(frame)
            frame_reasons.append(reason)

            # 진행 상황 업데이트
            pbar.update(1)

            # 체크포인트 저장 (10개 항목마다)
            if (i + 1) % 10 == 0 or i == len(df) - 1:
                temp_df = df.copy()
                temp_df['프레임 이름'] = frame_names + [''] * (len(temp_df) - len(frame_names))
                temp_df['이유'] = frame_reasons + [''] * (len(temp_df) - len(frame_reasons))
                temp_df.to_csv(checkpoint_path, encoding='utf-8-sig', index=False)
                logger.info(f"체크포인트 저장: {len(frame_names)}/{len(df)}")

        pbar.close()

        # 최종 결과 저장
        result_df = df.copy()
        result_df['프레임 이름'] = frame_names
        result_df['이유'] = frame_reasons

        # 프레임 분포 출력
        frame_distribution = result_df['프레임 이름'].value_counts()
        logger.info(f"EXAONE 32B 프레임 분포:\n{frame_distribution}")

        # 최종 결과 저장
        result_df.to_csv(output_path, encoding='utf-8-sig', index=False)
        logger.info(f"🎉 EXAONE 32B 분석 완료! 결과가 {output_path}에 저장되었습니다.")

        return result_df

# 텍스트 정규화 함수
def clean_text_control_chars_complete(text):
    """텍스트에서 제어문자 정리 + NaN 처리"""
    if pd.isna(text) or text == "":
        return "본문 정보 없음"

    # 문자열로 변환
    text = str(text)
    # 제어문자 제거/정규화
    text = text.replace('\n', ' ')       # 줄바꿈 → 공백
    text = text.replace('\r', ' ')       # 캐리지 리턴 → 공백
    text = text.replace('\t', ' ')       # 탭 → 공백
    text = text.replace('\b', ' ')       # 백스페이스 → 공백
    text = text.replace('\f', ' ')       # 폼피드 → 공백
    # 연속된 공백을 단일 공백으로 정리
    text = re.sub(r'\s+', ' ', text).strip()

    # 정리 후에도 빈 문자열이면 기본값 설정
    if text == "":
        return "본문 정보 없음"

    return text

# 빠른 테스트 함수 - GGUF 버전
def test_single_news_32b(text, client):
    """단일 뉴스 테스트용 함수 - EXAONE 32B GGUF"""
    analyzer = NewsFrameAnalyzer(client, output_dir='/mnt/c/Users/com/Downloads/')
    frame, reason = analyzer.analyze_news(text)

    print(f"뉴스: {text[:100]}...")
    print(f"🚀 EXAONE 32B 프레임: {frame}")
    print(f"🎯 분석 이유: {reason}")
    print("-" * 60)

    return frame, reason

# 메인 실행 코드
if __name__ == "__main__":
    # 🚀 EXAONE 32B GGUF 연결 설정
    print("🔷 EXAONE 32B GGUF 연결 중...")

    client = openai.OpenAI(
        base_url="http://121.132.230.74:1234/v1",  # LM Studio 주소(The local server is reachable at this address)
        api_key="not-needed"                       # LM Studio 주소 뒤에는 반드시 'v1' 붙이기.
    )

    # 연결 테스트
    try:
        test_response = client.chat.completions.create(
            model="exaone-3.5-32b-instruct",
            messages=[{"role": "user", "content": "Ai Risk란 무엇인가요? 128토큰 이내로 설명해주세요."}],
            max_tokens=128
        )
        print("✅ EXAONE 32B 연결 성공!")
    except Exception as e:
        print(f"❌ 연결 실패: {e}")
        exit(1)

    # 데이터 로드
    print("📊 데이터 로드 중...")

    # 파일 경로 수정 (Windows/WSL 경로) #"C:\Users\com\Desktop\ai컴기말\0. Data\3. semanticChunkeddata\chunk_only+오름차순.csv"
    file_path = "/mnt/c/Users/com/Desktop/ai컴기말/0. Data/3. semanticChunkeddata/test50_chunk_only+오름차순.csv"
    df = pd.read_csv(file_path, encoding="utf-8-sig")

    print("=== 원본 데이터 확인 ===")
    print(f"총 행 수: {len(df)}")
    print(f"text 열의 NaN 개수: {df['text'].isna().sum()}")
    print(f"text 열의 빈 문자열 개수: {(df['text'] == '').sum()}")

    # 텍스트 정제 - 개선된 함수 사용
    df['text'] = df['text'].apply(clean_text_control_chars_complete)
    print(f"데이터 로드 및 정제 완료: {len(df)}개의 기사")

    # 테스트 샘플 실행
    test_text = """
    특정 업무에서 인공지능(AI) 활용도가 높아짐에 따라 전 세계 기업의 41%가 2030년까지 고용 인력을 감축해나갈 계획인 것으로 나타났다.
    """

    print("=== 🚀 EXAONE 32B GGUF 테스트 실행 ===")
    test_single_news_32b(test_text, client)

    # 전체 분석 실행
    print("=== 🎯 EXAONE 32B 전체 데이터 분석 시작 ===")
    analyzer = NewsFrameAnalyzer(client, output_dir='/mnt/c/Users/com/Downloads/')

    result_df = analyzer.process_dataframe(
        df,
        output_file='AiRisk_frame_analysis_EXAONE32B_results_test3.csv' # 결과 파일 이름 수정
    )

    # 결과 요약
    frame_counts = result_df['프레임 이름'].value_counts()
    print("\n=== 🎉 EXAONE 32B 프레임 분석 결과 요약 ===")
    print(frame_counts)

    # 관련없음 비율 확인
    total_count = len(result_df)
    unrelated_count = frame_counts.get('관련없음', 0)
    success_rate = ((total_count - unrelated_count) / total_count) * 100
    print(f"\n🎯 EXAONE 32B 분류 성공률: {success_rate:.2f}% ({total_count - unrelated_count}/{total_count})")

    print("\n=== 🚀 EXAONE 32B GGUF 프레임 분석이 완료되었습니다! ===")

# 관련없음 예시 테스트 실행
if __name__ == "__main__":
    test_unrelated_examples()